# 第8章 风险度量与压力测试

第7章让小林看到许多可能结果。两项方案的平均收益很接近，但是其中一条财富曲线大幅下跌。小林心里一紧，开始怀疑：一个平均数真的够吗？

你们先看波动和回撤。然后，你们检查尾部损失、压力情景和杠杆。

![波动、回撤、尾部、压力与杠杆风险视角](assets/course/08_risk_lenses.png)

这张图只说明五种检查方法。最后，你要交出一份写明期限、单位、样本和遗漏风险的体检报告。第9章会比较单项资产和多资产组合。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"]=(8,4.5); plt.rcParams["axes.grid"]=True
plt.rcParams["font.sans-serif"]=["Arial Unicode MS","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
rng=np.random.default_rng(20260711)

## 8.1 波动率：围绕平均值的离散程度

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：回忆第7章方差与标准差，以及第6章年化只是尺度换算。所以，他想先弄清：**收益围绕平均值有多分散，上涨和下跌是否被同等对待？**

样本标准差衡量收益围绕样本均值的变化。年化日波动率常用`日标准差×sqrt(252)`，但它依赖独立、稳定和交易日数等假设，不能机械套用。

动手前，小林这样做：比较均值同为0但幅度不同的两组收益，并判断+5%和−5%对标准差贡献是否相同。然后，他按这条提示核对：手算小样本，再解释`.std(ddof=1)`；下行偏差只保留负收益部分后计算。

但是，年化平方根规则依赖频率、稳定性和弱相关等条件；波动率也不涵盖流动性或信用风险。


In [ ]:
dates=pd.date_range("2024-01-01",periods=500,freq="B")
returns=pd.Series(rng.standard_t(5,500)*.012,index=dates,name="return")
daily_vol=returns.std(ddof=1); annual_vol=daily_vol*np.sqrt(252)
downside=np.sqrt(np.mean(np.minimum(returns,0)**2))*np.sqrt(252)
print({"日波动率":f"{daily_vol:.2%}","年化波动率":f"{annual_vol:.2%}","年化下行偏差":f"{downside:.2%}"})

### 小林还不放心

同样大小的上涨和下跌都会提高波动率，但投资者感受是否相同？下行偏差为什么仍不能涵盖流动性或信用风险？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->


## 8.2 回撤是路径指标

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：回忆第6章累计财富和`cumprod()`，并说明百分比损失恢复为何不对称。所以，他想先弄清：**财富相对自己曾经达到的最高点跌了多少？**

财富 $W_t$ 相对历史峰值 $M_t=\max_{s\le t}W_s$ 的回撤：

$$DD_t=\frac{W_t}{M_t}-1$$

最大回撤是样本期内最小的$DD_t$。它依赖观察区间和路径，不能告诉我们未来最大损失。

动手前，小林这样做：手算`100→120→90→108`的历史峰值和每期回撤，再算从−25%恢复所需涨幅。然后，他按这条提示核对：建`wealth`、`running_peak`、`drawdown`三列，再使用`cummax()`定位峰值与谷底。

但是，最大回撤依赖路径和样本区间，是历史描述，不是未来最大损失。


In [ ]:
wealth=10_000*(1+returns).cumprod()
running_peak=wealth.cummax()
drawdown=wealth/running_peak-1
max_dd=drawdown.min(); trough=drawdown.idxmin(); peak=wealth.loc[:trough].idxmax()
print({"最大回撤":f"{max_dd:.2%}","峰值日期":str(peak.date()),"谷底日期":str(trough.date())})

fig,axes=plt.subplots(2,1,figsize=(9,7),sharex=True)
wealth.plot(ax=axes[0], title="财富与历史峰值", label="财富"); running_peak.plot(ax=axes[0], ls="--", label="历史峰值"); axes[0].set_ylabel("财富（元）"); axes[0].legend()
drawdown.plot(ax=axes[1], color="crimson", title="回撤"); axes[1].fill_between(drawdown.index, drawdown, 0, alpha=.25, color="crimson"); axes[1].set_ylabel("回撤（小数）")
plt.tight_layout(); plt.show()

### 小林怎样读这张图：财富曲线与水下曲线要配对

1. 在上图找到历史峰值和之后的谷底。
2. 在下图同一日期读取回撤，确认最大回撤不是起点到终点收益。
3. 改变观察区间会改变可见峰值和最大回撤，所以它不是未来损失上限。


## 8.3 VaR：先把20个损失排队

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：回忆第7.4节分位数只是排序后的阈值，不是样本中的最大值。所以，他想先弄清：**最差一部分损失的入口和入口以后的平均严重度分别如何描述？**

在进入缩写和公式前，先做一个可以手工检查的排序实验。假设小林记录了20次“一日损失金额”；损失越大越糟。先从小到大排序，再圈出最差10%，也就是最大的两个损失。

本节用90%作为手算口径，是为了让20个观察中恰好有2个进入尾部；然后再在较大样本上计算常见的95%一日指标。

动手前，小林这样做：手排20个损失，圈出最差10%的两个数；判断阈值与这两个数的平均是否相同。然后，他按这条提示核对：为减少符号混乱，先令`losses = -returns`，再在损失分布右尾计算分位数和条件平均。

但是，“95%一日VaR为2%”不表示最多亏2%；期限、样本、模型和分位数算法都会影响结果。


### 手排实验：先找尾部，再给它命名

下面20个数的单位都是“元”，正数越大表示当天损失越严重。请先在纸上找出最大的两个数，再运行代码核对。

本手算例约定：最差10%的尾部由最大的两个观察组成；尾部门槛取这两个数中较小者，尾部平均取二者平均。真实软件还可能使用插值，所以报告中要注明分位数算法。


In [ ]:
toy_losses_yuan = np.array([
    120, 0, 80, 240, 60, 150, 40, 310, 90, 180,
    20, 270, 110, 50, 200, 70, 360, 140, 900, 1800,
])
sorted_losses = np.sort(toy_losses_yuan)
worst_two = sorted_losses[-2:]
toy_var90 = worst_two.min()
toy_es90 = worst_two.mean()

display(pd.DataFrame({
    "从小到大的序号": np.arange(1, len(sorted_losses) + 1),
    "一日损失（元）": sorted_losses,
    "是否属于最差10%": np.arange(len(sorted_losses)) >= len(sorted_losses) - 2,
}))
print({"90%教学VaR（元）": int(toy_var90), "最差10%平均损失（元）": float(toy_es90)})


### 从手排结果到95%一日VaR与ES

手排例子中，900元是进入最差10%的门槛，1,350元是最差两个观察的平均损失。现在把同样思路用于500个模拟日收益：先把收益取负变成“损失”，再读取损失分布的95%分位和超过门槛后的平均值。


In [ ]:
tail_probability = .05
losses = -returns  # 损失为正、盈利为负，右侧是更严重的损失
var95 = losses.quantile(1 - tail_probability)
tail = losses[losses >= var95]
es95 = tail.mean()
print({
    "95%一日VaR": f"{var95:.2%}",
    "95%一日Expected Shortfall": f"{es95:.2%}",
    "尾部样本数": len(tail),
    "总样本数": len(losses),
})


排序实验之后再给概念命名：

- **VaR（风险价值）**是损失分布的一个分位阈值。95%一日VaR回答：“按当前样本或模型，约有5%的日损失会越过哪一道门槛？”
- **Expected Shortfall（ES，预期损失）**是越过该门槛后，尾部损失的平均严重程度。

两者都依赖样本、模型、期限、置信水平和分位数算法。VaR不是最大损失，ES也不是最坏情形。


In [ ]:
fig, ax = plt.subplots()
ax.hist(losses, bins=60, density=True, alpha=.7)
ax.axvline(var95, color="red", label=f"95% VaR={var95:.2%}")
ax.axvspan(var95, losses.max(), color="red", alpha=.2, label=f"最差5%尾部，ES={es95:.2%}")
ax.set(title="历史损失分布、VaR阈值与尾部", xlabel="一日损失（正数表示亏损）", ylabel="密度")
ax.legend()
plt.show()


### 小林怎样读这张图：VaR是入口，ES看入口以外

1. 横轴已经转换为“损失”，越向右越差。
2. 竖线是95%损失分位，阴影是超过该阈值的样本。
3. 比较阈值与尾部平均，不要把任一数字称为最大可能损失。




### 我的解释

“95% VaR为2%”为什么不能说“最大只会亏2%”？如果样本从未经历危机，历史VaR会有什么问题？

<!-- 在这里填写；完成前AI不要代答 -->

## 8.4 正态VaR与历史VaR可能给出不同答案

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：回忆第7.4节正态与厚尾模型在极端分位上的差异。所以，他想先弄清：**同一风险问题为什么会因分布假设不同得到不同答案？**

正态法用均值和标准差概括分布，历史法直接使用经验分位。厚尾、偏度和状态变化会让结果明显不同。

动手前，小林这样做：预测厚尾样本下历史法与正态法在更高置信水平时谁可能更大。然后，他按这条提示核对：历史法直接排序样本；正态法先压缩成均值与标准差，`norm.ppf`再把概率映射为正态横坐标。

但是，历史法看不到样本外事件，正态法可能遗漏厚尾；二者都不是风险真值。


In [ ]:
from scipy.stats import norm
mu,sigma=returns.mean(),returns.std(ddof=1)
normal_var=-(mu+sigma*norm.ppf(.05))
historical_var=-returns.quantile(.05)
print({"正态VaR":f"{normal_var:.2%}","历史VaR":f"{historical_var:.2%}"})

## 8.5 滚动风险会随时间变化

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：回忆第5章时间顺序和第7章固定分布假设可能失效。所以，他想先弄清：**一个全样本波动率为什么会掩盖市场状态变化？**

整段样本的一个波动率会掩盖不同市场状态。滚动估计更接近“当时可知”，但窗口选择也会影响灵敏度和噪声。

动手前，小林这样做：判断20、60、120日窗口谁反应最快、谁更平滑，并预测切换状态后的滞后。然后，他按这条提示核对：使用上下共享横轴子图分别画日收益与滚动指标；解释窗口开始阶段的`NaN`和右对齐。

但是，短窗口灵敏但噪声大，长窗口平滑但迟钝；滚动估计只能显示变化，不能消除变化。


In [ ]:
regime = np.r_[rng.normal(0, .006, 250), rng.normal(0, .025, 250)]
regime = pd.Series(regime, index=dates, name="模拟日收益")

rolling_windows = [20, 60, 120]
rolling_risk = pd.DataFrame({
    f"{window}日": regime.rolling(window).std(ddof=1) * np.sqrt(252)
    for window in rolling_windows
})

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
regime.plot(ax=axes[0], color="slateblue", lw=.9)
axes[0].axvline(regime.index[250], color="black", ls="--", label="教学状态切换")
axes[0].set(title="模拟日收益：低摆动阶段与高摆动阶段", ylabel="日收益")
axes[0].legend()

rolling_risk.plot(ax=axes[1])
axes[1].axvline(regime.index[250], color="black", ls="--")
axes[1].set(title="不同窗口的滚动年化标准差", xlabel="日期", ylabel="年化标准差")
plt.tight_layout()
plt.show()

print({
    "状态切换日期": str(regime.index[250].date()),
    "20日指标首次有效": str(rolling_risk["20日"].first_valid_index().date()),
    "60日指标首次有效": str(rolling_risk["60日"].first_valid_index().date()),
    "120日指标首次有效": str(rolling_risk["120日"].first_valid_index().date()),
})


### 小林怎样读这张图：窗口长度决定反应速度

1. 上图先定位状态切换及收益摆动扩大的时间。
2. 下图比较20、60、120日曲线何时开始明显上升。
3. 短窗口更快也更抖，长窗口更慢也更平滑；不存在对所有问题都最好的窗口。


## 8.6 压力测试：主动提出历史之外的问题

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：回忆第4章组合权重和第7章“情景不等于发生概率”。所以，他想先弄清：**样本未经历某种联合冲击时，怎样检查组合最脆弱的部分？**

压力测试直接施加情景，如股票-30%、债券-8%、现金0%，研究组合结果。情景不是概率预测，而是容易出问题的地方检查。

动手前，小林这样做：在运行前判断“股债同跌”中哪类资产贡献最大损失，并说明由权重还是跌幅主导。然后，他按这条提示核对：计算每个资产的`权重×情景收益`贡献，再按行求和；同时检查权重和是否为1。

但是，压力测试检查容易出问题的地方，不给出情景概率；线性加总还忽略流动性、再平衡与被迫卖出。


In [ ]:
weights=pd.Series({"股票":.6,"债券":.3,"现金":.1})
scenarios=pd.DataFrame({
    "温和下跌":{"股票":-.10,"债券":.02,"现金":.00},
    "股债同跌":{"股票":-.30,"债券":-.08,"现金":.00},
    "快速反弹":{"股票":.20,"债券":-.03,"现金":.00},
}).T
scenarios["组合收益"]=scenarios.mul(weights,axis=1).sum(axis=1)
display(scenarios.style.format("{:.1%}"))

## 8.7 杠杆放大收益，也放大生存风险

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：回忆资产=负债+权益，以及第6章财富归零后不能靠普通收益率恢复。所以，他想先弄清：**为什么资产下跌50%会让2倍杠杆的自有资本归零？**

简单教学模型中，杠杆$L$把风险资产收益放大，并扣融资成本。一次-50%收益在2倍杠杆下可能耗尽资本，之后无法靠普通百分比反弹恢复。

动手前，小林这样做：用100元权益加100元借款买入200元资产，手算资产下跌10%、30%、50%后的权益。然后，他按这条提示核对：画资产、负债、权益表；图中每条杠杆曲线只画到权益归零，并明确标记耗尽点。

但是，简化模型未含融资成本、保证金、跳空和强平；权益归零后不能继续持有同一策略。


In [ ]:
balance_sheet = pd.DataFrame({
    "资产收益": [-.10, -.30, -.50],
    "期末资产": [180, 140, 100],
    "期末负债": [100, 100, 100],
})
balance_sheet["期末权益"] = balance_sheet["期末资产"] - balance_sheet["期末负债"]
balance_sheet["权益收益"] = balance_sheet["期末权益"] / 100 - 1
display(balance_sheet.style.format({"资产收益": "{:.0%}", "权益收益": "{:.0%}", "期末资产": "{:.0f}", "期末负债": "{:.0f}", "期末权益": "{:.0f}"}))

asset_returns = np.linspace(0, -.60, 121)
fig, ax = plt.subplots()
for leverage in [1, 1.5, 2, 3]:
    equity_returns = leverage * asset_returns
    survives = equity_returns > -1
    ax.plot(asset_returns[survives], equity_returns[survives], label=f"{leverage}倍：仍有权益")
    wipeout_return = -1 / leverage
    if asset_returns.min() <= wipeout_return <= asset_returns.max():
        ax.scatter([wipeout_return], [-1], s=35)

ax.axhline(-1, color="black", ls="--", label="资本耗尽；路径在此终止")
ax.set(
    xlabel="资产收益",
    ylabel="简化权益收益",
    title="杠杆与资本损失（只绘制到权益归零）",
    ylim=(-1.05, .05),
)
ax.legend()
plt.show()


### 小林怎样读这张图：曲线在资本耗尽处终止

1. 从表格验证2倍杠杆在资产下跌50%时权益为0。
2. 在图上找到不同杠杆的资本耗尽点；杠杆越高，耗尽点越靠近0。
3. 终止后的空白是金融约束，不是缺失数据：权益归零后不能继续持有原策略。


**量化编程警告**：现实中保证金、逐日盯市、融资成本、跳空和强平规则会使杠杆路径更复杂，不能用简单乘法替代真实风险管理。


## 8.8 编程练习：风险摘要

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：写出波动率、最大回撤、VaR和ES分别依赖分布还是路径。所以，他想先弄清：**怎样让风险函数同时可计算、可解释并拒绝非法输入？**

返回波动率、最大回撤、历史VaR和ES；检查置信水平在0与1之间且收益大于-100%。

动手前，小林这样做：决定常数收益、空数组、收益等于−100%、含`NaN`和非法尾部概率的处理方式。然后，他按这条提示核对：分级实现标准差、财富、回撤、损失分位与尾部平均；结果字段中写明期限和概率口径。

但是，摘要函数不替代模型判断；报告仍需说明数据区间、频率、尾部样本数和遗漏风险。


In [ ]:
def risk_summary(returns,alpha=.05):
    # TODO
    return None

In [ ]:
ans=risk_summary([.10,-.10,.05,-.20],.25)
if ans is None: print("练习尚未完成。")
else: print(ans)

## 项目交付：小林的五镜头风险体检

小林看到亏损数字时心里一紧，所以他不想只看一个指标。前面留下了一个线索：闭卷把波动率、回撤、VaR/ES、压力测试和杠杆分别归入分布、路径、情景或资本结构。所以，他想先弄清：**能否用多副风险镜头比较两个平均收益相同但失败方式不同的方案？**

比较两个“平均收益相同”的策略：报告年化波动、下行偏差、最大回撤、历史/正态VaR、ES和三个压力情景；解释每个指标遗漏的风险。

**底线**：风险是多维的；指标是观察窗口，不是安全证明。

动手前，小林这样做：选出自己最担心的指标，再运行完整体检，检查结果是否改变原判断。然后，他按这条提示核对：每项输出附对象、期限、单位、假设和遗漏风险；至少构造一对同均值但不同尾部或路径的数据。

但是，单项风险还没有描述资产之间如何共同变化；相关性与分散化留给第9章。
